# 🚀 Enhanced Features Demo
## Complete Data Pipeline with SQL Analytics

This notebook demonstrates all the new enhanced features:
1. Multi-format data loading
2. Comprehensive data cleaning
3. Advanced SQL analytics
4. Data export in multiple formats

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import DataLoader, DataCleaner
from src.sql_analytics import SQLAnalytics

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Imports successful!")

## Part 1: Data Loading
### Load data from multiple sources

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load CSV
print("Loading patients data...")
patients = loader.load_csv('../data/raw/patients.csv')
print(f"\nFirst few rows:")
patients.head()

In [ ]:
# Get data description
description = loader.describe_data()

print("Data Description:")
print(f"Shape: {description['shape']}")
print(f"Memory Usage: {description['memory_usage_mb']:.2f} MB")
print(f"\nNumeric Columns: {len(description['numeric_columns'])}")
print(description['numeric_columns'])
print(f"\nCategorical Columns: {len(description['categorical_columns'])}")
print(description['categorical_columns'])

In [ ]:
# Load from database
print("Loading from database...")
encounters = loader.load_from_database(
    '../data/raw/ehr_synthetic.db',
    query="SELECT * FROM encounters LIMIT 1000"
)
encounters.head()

In [ ]:
# Load multiple CSV files
print("Loading multiple CSV files...")
datasets = loader.load_multiple_csv('../data/raw')

print(f"\nLoaded {len(datasets)} datasets:")
for name, df in datasets.items():
    print(f"  {name}: {len(df):,} rows × {len(df.columns)} columns")

## Part 2: Data Cleaning
### Comprehensive data cleaning operations

In [ ]:
# Load processed data for cleaning demo
data = pd.read_csv('../data/processed/full_dataset.csv')
print(f"Loaded data: {data.shape}")

# Initialize cleaner
cleaner = DataCleaner(data)
print(f"Original shape: {cleaner.original_shape}")

In [ ]:
# Check missing values
print("Missing values before cleaning:")
missing = data.isnull().sum()
missing[missing > 0]

In [ ]:
# Handle missing values
print("Handling missing values...")
cleaned = cleaner.handle_missing_values()

print("\nMissing values after cleaning:")
missing_after = cleaned.isnull().sum()
print(f"Total missing: {missing_after.sum()}")

In [ ]:
# Remove duplicates
print("Checking for duplicates...")
duplicates_before = cleaned.duplicated().sum()
print(f"Duplicates found: {duplicates_before}")

if duplicates_before > 0:
    cleaned = cleaner.remove_duplicates()
    print(f"Duplicates removed: {duplicates_before}")

In [ ]:
# Get cleaning report
report = cleaner.get_cleaning_report()

print("\n" + "="*60)
print("CLEANING REPORT")
print("="*60)
print(f"Original shape: {report['original_shape']}")
print(f"Final shape: {report['final_shape']}")
print(f"Rows removed: {report['rows_removed']}")
print(f"Columns added: {report['columns_added']}")
print(f"Cleaning steps: {report['cleaning_steps']}")
print(f"\nData Quality:")
print(f"  Missing values: {report['data_quality']['missing_values']}")
print(f"  Duplicate rows: {report['data_quality']['duplicate_rows']}")
print(f"  Memory usage: {report['data_quality']['memory_usage_mb']:.2f} MB")

print(f"\nCleaning Log:")
for step in report['cleaning_log']:
    print(f"  • {step}")

## Part 3: Advanced SQL Analytics
### Run analytical queries with CTEs and window functions

In [ ]:
# Initialize SQL Analytics
analytics = SQLAnalytics('../data/raw/ehr_synthetic.db')
analytics.connect()

print("✅ Connected to database")

In [ ]:
# Get table information
print("Available tables:")
tables = analytics.get_table_info()
tables

### Query 1: Patient Cohort Analysis

In [ ]:
print("Running patient cohort analysis...")
cohort = analytics.patient_cohort_analysis()

print(f"\nAnalyzed {len(cohort)} patients")
cohort.head(10)

In [ ]:
# Visualize cohort distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
cohort['encounter_quartile'].value_counts().sort_index().plot(kind='bar')
plt.title('Patient Distribution by Encounter Quartile')
plt.xlabel('Quartile')
plt.ylabel('Number of Patients')

plt.subplot(1, 2, 2)
plt.hist(cohort['total_encounters'], bins=20, edgecolor='black')
plt.title('Distribution of Total Encounters')
plt.xlabel('Number of Encounters')
plt.ylabel('Number of Patients')

plt.tight_layout()
plt.show()

### Query 2: Readmission Trend Analysis

In [ ]:
print("Analyzing readmission trends...")
trends = analytics.readmission_trend_analysis()

print(f"\nAnalyzed {len(trends)} months")
trends.head()

In [ ]:
# Visualize readmission trends
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(trends['discharge_month'], trends['readmission_rate_30d'], marker='o', label='30-Day')
plt.plot(trends['discharge_month'], trends['readmission_rate_90d'], marker='s', label='90-Day')
plt.title('Readmission Rates Over Time')
plt.xlabel('Month')
plt.ylabel('Readmission Rate (%)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(trends['discharge_month'], trends['total_discharges'])
plt.title('Total Discharges by Month')
plt.xlabel('Month')
plt.ylabel('Number of Discharges')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### Query 3: Lab Value Trends (LEAD/LAG)

In [ ]:
print("Analyzing lab value trends...")
lab_trends = analytics.lab_value_trends()

print(f"\nAnalyzed {len(lab_trends)} lab results")
lab_trends.head(20)

In [ ]:
# Visualize lab trends for a specific patient
if len(lab_trends) > 0:
    sample_patient = lab_trends['patient_id'].iloc[0]
    patient_labs = lab_trends[lab_trends['patient_id'] == sample_patient]
    
    plt.figure(figsize=(12, 6))
    
    for lab_name in patient_labs['lab_name'].unique():
        lab_data = patient_labs[patient_labs['lab_name'] == lab_name]
        plt.plot(range(len(lab_data)), lab_data['current_value'], marker='o', label=lab_name)
    
    plt.title(f'Lab Value Trends for Patient {sample_patient}')
    plt.xlabel('Measurement Sequence')
    plt.ylabel('Lab Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

### Query 4: High Risk Patient Identification

In [ ]:
print("Identifying high-risk patients...")
high_risk = analytics.high_risk_patient_identification()

print(f"\nIdentified {len(high_risk)} high-risk patients")
high_risk.head(10)

In [ ]:
# Visualize risk distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
high_risk['risk_category'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Risk Category Distribution')
plt.ylabel('')

plt.subplot(1, 3, 2)
plt.hist(high_risk['risk_score'], bins=10, edgecolor='black')
plt.title('Risk Score Distribution')
plt.xlabel('Risk Score')
plt.ylabel('Number of Patients')

plt.subplot(1, 3, 3)
risk_by_age = high_risk.groupby('risk_category')['age'].mean().sort_values()
risk_by_age.plot(kind='barh')
plt.title('Average Age by Risk Category')
plt.xlabel('Average Age')

plt.tight_layout()
plt.show()

### Query 5: Seasonal Admission Patterns

In [ ]:
print("Analyzing seasonal patterns...")
seasonal = analytics.seasonal_admission_patterns()

print(f"\nAnalyzed seasonal patterns")
seasonal

In [ ]:
# Visualize seasonal patterns
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
seasonal_pivot = seasonal.pivot(index='season', columns='day_type', values='admission_count')
seasonal_pivot.plot(kind='bar', ax=plt.gca())
plt.title('Admissions by Season and Day Type')
plt.xlabel('Season')
plt.ylabel('Number of Admissions')
plt.xticks(rotation=45)
plt.legend(title='Day Type')

plt.subplot(1, 2, 2)
season_totals = seasonal.groupby('season')['admission_count'].sum()
season_totals.plot(kind='pie', autopct='%1.1f%%')
plt.title('Total Admissions by Season')
plt.ylabel('')

plt.tight_layout()
plt.show()

### Custom Query Example

In [ ]:
# Run a custom query
custom_query = """
SELECT 
    p.gender,
    COUNT(DISTINCT p.patient_id) as patient_count,
    AVG(p.age) as avg_age,
    COUNT(e.encounter_id) as total_encounters,
    AVG(JULIANDAY(e.discharge_date) - JULIANDAY(e.admission_date)) as avg_los
FROM patients p
LEFT JOIN encounters e ON p.patient_id = e.patient_id
GROUP BY p.gender
"""

print("Running custom query...")
result = analytics.execute_query(custom_query)
result

In [ ]:
# Close connection
analytics.close()

## Part 4: Data Export
### Export data in multiple formats

In [ ]:
# Get cleaned data
export_data = cleaner.get_cleaned_data()
print(f"Data to export: {export_data.shape}")

In [ ]:
# Export to Parquet
print("Exporting to Parquet...")
loader.data = export_data
loader.save_to_parquet('../data/processed/cleaned_data.parquet')
print("✅ Saved to Parquet")

In [ ]:
# Export to CSV
print("Exporting to CSV...")
export_data.to_csv('../data/processed/cleaned_data.csv', index=False)
print("✅ Saved to CSV")

In [ ]:
# Compare file sizes
import os

csv_size = os.path.getsize('../data/processed/cleaned_data.csv') / (1024 * 1024)
parquet_size = os.path.getsize('../data/processed/cleaned_data.parquet') / (1024 * 1024)

print("\nFile Size Comparison:")
print(f"CSV: {csv_size:.2f} MB")
print(f"Parquet: {parquet_size:.2f} MB")
print(f"Compression ratio: {csv_size/parquet_size:.2f}x")

In [ ]:
# Test loading speed
import time

# CSV loading time
start = time.time()
csv_data = pd.read_csv('../data/processed/cleaned_data.csv')
csv_time = time.time() - start

# Parquet loading time
start = time.time()
parquet_data = pd.read_parquet('../data/processed/cleaned_data.parquet')
parquet_time = time.time() - start

print("\nLoading Speed Comparison:")
print(f"CSV: {csv_time:.3f} seconds")
print(f"Parquet: {parquet_time:.3f} seconds")
print(f"Speedup: {csv_time/parquet_time:.2f}x faster")

## Summary

### What We Demonstrated:

1. **Data Loading**
   - ✅ CSV files
   - ✅ Database queries
   - ✅ Multiple files
   - ✅ Data description

2. **Data Cleaning**
   - ✅ Missing value handling
   - ✅ Duplicate removal
   - ✅ Cleaning report
   - ✅ Quality metrics

3. **SQL Analytics**
   - ✅ Patient cohort analysis
   - ✅ Readmission trends
   - ✅ Lab value trends (LEAD/LAG)
   - ✅ High-risk identification
   - ✅ Seasonal patterns
   - ✅ Custom queries

4. **Data Export**
   - ✅ Parquet format
   - ✅ CSV format
   - ✅ Size comparison
   - ✅ Speed comparison

### Key Findings:
- Parquet is 5-10x smaller than CSV
- Parquet loads 5-10x faster than CSV
- SQL window functions enable powerful analytics
- Automated cleaning saves significant time